In [ ]:

# ============================================================
# Environment + Imports (Prediction/Ensemble v3)
# ============================================================
import os, sys, time, gc, warnings, json, zipfile, hashlib, traceback
warnings.filterwarnings("ignore")

# max_split_size_mb:9000 — the cuDNN PRECOMP_GEMM workspace for Conv3d(96ch, 128³) is
# 10.12 GiB. Without this setting, PyTorch's caching allocator splits that block to service
# smaller requests (model weights 191 MB, encoder skip-connection tensors ~700 MB), leaving
# only 9.86 GiB of fragmented free blocks — not enough for the next model's workspace.
# By marking blocks >9 GB as unsplittable, the 10.12 GB block stays intact in the cache
# and is cleanly reused for all 9 models (zero cudaMalloc after the first model).
# Small allocations go to cudaMalloc from the ~4.4 GB of free physical VRAM instead.
# Do NOT use expandable_segments:True — it uses CUDA VMM which corrupts the T4 context.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:9000")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_ckpt_fn

import scipy.ndimage as ndi
from scipy.ndimage import (distance_transform_edt, generate_binary_structure,
                           label as cc_label, binary_erosion, binary_dilation,
                           binary_opening, gaussian_filter)

try:
    import tifffile
    _HAVE_TIFFFILE = True
except ImportError:
    _HAVE_TIFFFILE = False

def _read_tif_pil(path):
    from PIL import Image
    img = Image.open(path)
    frames = []
    try:
        while True:
            frames.append(np.array(img))
            img.seek(img.tell() + 1)
    except EOFError:
        pass
    return np.stack(frames, axis=0)

def read_tif(path):
    # tifffile preferred; fall back to PIL for LZW-compressed TIFFs.
    # PIL handles LZW natively with no extra packages — works offline.
    if _HAVE_TIFFFILE:
        try:
            return tifffile.imread(path)
        except Exception:
            return _read_tif_pil(path)
    return _read_tif_pil(path)

def write_tif(path, data):
    if _HAVE_TIFFFILE:
        tifffile.imwrite(path, data)
    else:
        from PIL import Image as PILImage
        imgs = [PILImage.fromarray(data[z]) for z in range(data.shape[0])]
        imgs[0].save(path, save_all=True, append_images=imgs[1:])

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[ENV] PyTorch {torch.__version__}, Device: {DEVICE}")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"[ENV] GPU: {props.name}, VRAM: {props.total_memory / 1e9:.1f} GB")
    # benchmark=False (heuristic, default): cuDNN picks algorithm via lookup table.
    # For Conv3d(96ch, 128³) on T4, heuristic selects PRECOMP_GEMM (10.12 GB workspace).
    # This is fine — max_split_size_mb:9000 keeps the 10.12 GB block intact in cache
    # so the workspace is reused cleanly across all 9 models with no re-allocation.
    # Using benchmark=True would pollute the cache with many small blocks from algo tests.
    torch.backends.cudnn.benchmark = False
    print("[ENV] cuDNN benchmark=False (heuristic algo selection; workspace reused via max_split_size_mb:9000)")

T_START = time.time()
def elapsed_h():
    return (time.time() - T_START) / 3600.0
def budget_ok(max_h=8.5):
    return elapsed_h() < max_h


In [ ]:

# ============================================================
# Configuration v3 + Explicit Model Paths
# ============================================================
NUM_CLASSES = 2
IGNORE_LABEL = 255
IN_CHANNELS = 1
FEATURES = [32, 64, 128, 256, 320, 320]
BLOCKS = [1, 3, 4, 6, 6, 6]
STRIDES = [[1,1,1],[2,2,2],[2,2,2],[2,2,2],[2,2,2],[2,2,2]]
DEC_CONVS = [1, 1, 1, 1, 1]

ROOT_CANDS = [
    "/kaggle/input/competitions/vesuvius-challenge-surface-detection",
    "/kaggle/input/vesuvius-challenge-surface-detection",
]
ROOT_DIR = next((p for p in ROOT_CANDS if os.path.exists(p)), None)
if ROOT_DIR is None:
    ROOT_DIR = "/kaggle/input/competitions/vesuvius-challenge-surface-detection"
TEST_DIR = os.path.join(ROOT_DIR, "test_images")

MODEL_DATASET_MAP = {
    # Super model X: single unified model; loads all available per-metric checkpoints
    # as separate ensemble variants (best_score/surfdice/voi/topo + swa = up to 5 variants).
    "model_x": {
        "dataset_slugs": ["vesuvius-v3-checkpoints", "vesuvius-model-x", "vesuvius-model-x-ckpt"],
        "ckpt_candidates": ["model_x_best_score.pt", "model_x_best_surfdice.pt",
                            "model_x_best_voi.pt", "model_x_best_topo.pt",
                            "model_x_best_loss.pt",
                            "model_x_late60.pt", "model_x_late70.pt",
                            "model_x_late80.pt", "model_x_late90.pt",
                            "model_x_swa.pt", "model_x_best.pt"],
        "meta": "model_x_meta.json",
    },
}

MODEL_PATHS = {}
META_PATHS = {}

# Multi-checkpoint ensemble: load ALL available per-metric checkpoints as separate ensemble members.
# 1 trained model → up to 4 variants. 3 models → up to 12 variants. Zero extra training cost.
# Variants have different weight optima (topo-specialist ckpt ≠ surfdice-specialist ckpt).
# Weight multipliers downweight metric-specific checkpoints vs. composite-optimal checkpoint.
CKPT_TYPE_WEIGHTS = {
    "best_score":    1.00,  # composite-optimal — full weight
    "best_surfdice": 0.70,  # surfdice-optimised — slight downweight (may sacrifice topo/voi)
    "best_topo":     0.70,  # topo-optimised — slight downweight (may sacrifice sd/voi)
    "best_voi":      0.60,  # voi-optimised — lowest (voi is already strong at epoch 4: 0.866)
    "best_loss":     0.60,  # best training loss — no val filtering, more conservative weight
    "late60":        0.65,  # late snapshot at 60% training time — early late-phase
    "late70":        0.65,  # late snapshot at 70% training time
    "late80":        0.65,  # late snapshot at 80% training time (SWA start)
    "late90":        0.65,  # late snapshot at 90% training time — near end
    "swa":           0.75,  # stochastic weight average — smooth ensemble of late-phase weights
    "best":          0.80,  # legacy fallback key
}
CKPT_MULTS = {}  # compound_key → weight multiplier

# Build dataset search roots dynamically.
# Kaggle mounts user datasets at /kaggle/input/datasets/{username}/ (not /kaggle/input/).
# Also check standard /kaggle/input/ for backwards compat and -ckpt slug fallbacks.
_DATASET_SEARCH_ROOTS = ["/kaggle/input"]
_ds_users_base = "/kaggle/input/datasets"
if os.path.exists(_ds_users_base):
    for _uname in sorted(os.listdir(_ds_users_base)):
        _upath = os.path.join(_ds_users_base, _uname)
        if os.path.isdir(_upath):
            _DATASET_SEARCH_ROOTS.append(_upath)
print(f"[CFG] Dataset search roots: {_DATASET_SEARCH_ROOTS}")

for mname, spec in MODEL_DATASET_MAP.items():
    found_any = False
    for _root in _DATASET_SEARCH_ROOTS:
        for slug in spec["dataset_slugs"]:
            slug_base = os.path.join(_root, slug)
            if not os.path.exists(slug_base):
                continue
            # Handle optional version subdirectory: slug/1/, slug/2/ (Kaggle versioned datasets)
            _ver_dirs = [""]
            for _e in sorted(os.listdir(slug_base), reverse=True):
                if os.path.isdir(os.path.join(slug_base, _e)) and _e.isdigit():
                    _ver_dirs.insert(0, _e)
            for _ver in _ver_dirs:
                ver_base = os.path.join(slug_base, _ver) if _ver else slug_base
                for sub in ["checkpoints", ""]:
                    base = os.path.join(ver_base, sub) if sub else ver_base
                    meta_path = os.path.join(base, spec["meta"])
                    for ckpt_name in spec.get("ckpt_candidates", [f"{mname}_best.pt"]):
                        ckpt = os.path.join(base, ckpt_name)
                        if os.path.exists(ckpt):
                            ckpt_stem = ckpt_name.replace(".pt", "")
                            ckpt_type = ckpt_stem[len(f"{mname}_"):] if ckpt_stem.startswith(f"{mname}_") else "best"
                            compound_key = f"{mname}_{ckpt_type}"
                            if compound_key not in MODEL_PATHS:
                                MODEL_PATHS[compound_key] = ckpt
                                CKPT_MULTS[compound_key] = CKPT_TYPE_WEIGHTS.get(ckpt_type, 0.80)
                                if os.path.exists(meta_path):
                                    META_PATHS[compound_key] = meta_path
                                found_any = True
    if not found_any:
        print(f"[WARN] {mname} not found in any of {spec['dataset_slugs']}")

assert len(MODEL_PATHS) > 0, "FATAL: No model checkpoints found!"

print(f"[CFG] Loaded {len(MODEL_PATHS)} checkpoint variant(s) for ensemble:")
for k, v in MODEL_PATHS.items():
    sz = os.path.getsize(v) / 1e6
    mult = CKPT_MULTS.get(k, 1.0)
    print(f"  {k}: {v} ({sz:.1f} MB, weight_mult={mult:.2f})")

DEFAULT_ROI = (128, 128, 128)  # trained at 128³ max; larger patches are OOD and OOM
INF_OVERLAP = 0.50   # 0.50→125 patches/vol (was 0.75→343 patches/vol which timed out on 120 vols)
USE_TTA = True
MAX_PRED_HOURS = 8.5
ENS_MODE = "logit_weighted"

# v3: Role weights for ensemble (balanced > topology > surface)
ROLE_WEIGHTS = {"generalist": 1.4, "anti_merge": 1.1, "surface": 1.3, "balanced": 1.0}
CONF_TEMPERATURE = 0.80

# v3: Post-processing (updated defaults)
PP_DUST_MIN_3D = 192
PP_HOLE_MAX_2D = 128   # was 32 — fill larger surface through-holes per Z-slice
PP_OPEN_R_XY = 1
PP_OPEN_R_3D = 1
PP_SMOOTH_SIGMA = 0.3
PP_BRIDGE_KILL = True
PP_BK_NECK_R = 1
PP_BK_MIN_LOBE = 10000
PP_BK_MIN_NECK_LEN = 2
PP_EMPTY_TL_DROP = 0.08
PP_EMPTY_TH_DROP = 0.10
PP_OVERFULL_FRAC = 0.45
PP_OVERFULL_TH_RAISE = 0.05
DEFAULT_TL = 0.34
DEFAULT_TH = 0.62
AUTO_DROP_THRESH = 0.15
# TTA only on difficult volumes (saves 7x overhead on easy ones). 0.35 = ~top 30% of volumes.
TTA_DIFFICULTY_THRESH = 0.35
# Upgrade 4: Pathology gate — bridge-kill only if CC count above this
PP_PATHOLOGY_CC_THRESH = 8

# Global inference patch cap — mutable list so OOM retry can reduce it without
# passing it through the thread closure boundary.  Starts at 128³ (max quality).
# On CUDA OOM the main loop reduces this and retries the same volume.
# Ladder: 128 → 96 → 64 (each step is ~56% memory reduction for cubic patches).
_INF_PATCH_CAP_GLOBAL = [128, 128, 128]

# §1 ID contract: test.csv is the authoritative source of test IDs.
# Old notebook design: never hard-code IDs, never use directory listing alone.
# The actual hidden test set has ~120 volumes; test.csv lists every one.
_test_csv_path = os.path.join(ROOT_DIR, "test.csv")
if os.path.exists(_test_csv_path):
    import pandas as _pd
    _test_df = _pd.read_csv(_test_csv_path)
    _csv_ids = [str(i) for i in _test_df["id"].tolist()]
    # Deduplicate while preserving CSV order
    _seen_ids = set()
    test_ids = [i for i in _csv_ids if i not in _seen_ids and not _seen_ids.add(i)]
    test_paths = [os.path.join(TEST_DIR, f"{i}.tif") for i in test_ids]
    _on_disk = [os.path.exists(p) for p in test_paths]
    _missing = [i for i, ok in zip(test_ids, _on_disk) if not ok]
    if _missing:
        print(f"[WARN] {len(_missing)}/{len(test_ids)} test IDs in CSV have no .tif on disk: {_missing[:5]}")
    print(f"[CFG] test.csv → {len(test_ids)} IDs, {sum(_on_disk)} found on disk")
else:
    # Fallback: directory scan (backward compat for test runs without test.csv)
    print("[WARN] test.csv not found — falling back to TEST_DIR directory scan")
    test_paths = sorted([os.path.join(TEST_DIR, f)
                         for f in os.listdir(TEST_DIR) if f.endswith(".tif")]) \
                 if os.path.isdir(TEST_DIR) else []
    test_ids = [os.path.splitext(os.path.basename(p))[0] for p in test_paths]
print(f"[CFG] {len(test_ids)} test volumes, fusion={ENS_MODE}")


In [ ]:

# ============================================================
# Architecture (inference mode — no aux heads needed)
# ============================================================

class ConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, bias=True):
        super().__init__()
        if isinstance(kernel_size, int): kernel_size = [kernel_size] * 3
        if isinstance(stride, int): stride = [stride] * 3
        padding = [k // 2 for k in kernel_size]
        self.conv = nn.Conv3d(in_ch, out_ch, kernel_size, stride=stride, padding=padding, bias=bias)
        self.norm = nn.InstanceNorm3d(out_ch, eps=1e-5, affine=True)
        self.act = nn.LeakyReLU(inplace=True)
    def forward(self, x):
        return self.act(self.norm(self.conv(x)))

class ResBlock3D(nn.Module):
    def __init__(self, ch, kernel_size=3, bias=True):
        super().__init__()
        p = kernel_size // 2
        self.conv1 = nn.Conv3d(ch, ch, kernel_size, padding=p, bias=bias)
        self.norm1 = nn.InstanceNorm3d(ch, eps=1e-5, affine=True)
        self.conv2 = nn.Conv3d(ch, ch, kernel_size, padding=p, bias=bias)
        self.norm2 = nn.InstanceNorm3d(ch, eps=1e-5, affine=True)
        self.act = nn.LeakyReLU(inplace=True)
    def forward(self, x):
        r = x
        x = self.act(self.norm1(self.conv1(x)))
        x = self.norm2(self.conv2(x))
        return self.act(x + r)

class ResidualEncoderUNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=2,
                 features=(32,64,128,256,320,320), blocks=(1,3,4,6,6,6),
                 strides=((1,1,1),(2,2,2),(2,2,2),(2,2,2),(2,2,2),(2,2,2)),
                 dec_convs=(1,1,1,1,1), deep_supervision=False, use_grad_ckpt=False,
                 aux_heads=False):
        super().__init__()
        self.deep_supervision = deep_supervision
        self._ckpt = use_grad_ckpt
        self.aux_heads = aux_heads
        n = len(features)
        self.enc = nn.ModuleList()
        for s in range(n):
            in_c = in_channels if s == 0 else features[s-1]
            out_c = features[s]
            st = list(strides[s]) if isinstance(strides[s], (list,tuple)) else [strides[s]]*3
            layers = [ConvBlock3D(in_c, out_c, 3, stride=st)]
            for _ in range(blocks[s]-1):
                layers.append(ResBlock3D(out_c, 3))
            self.enc.append(nn.Sequential(*layers))
        self.up = nn.ModuleList()
        self.dec = nn.ModuleList()
        self.seg = nn.ModuleList()
        for i in range(n-1):
            s = n-1-i
            enc_ch, skip_ch, out_ch = features[s], features[s-1], features[s-1]
            st = list(strides[s]) if isinstance(strides[s], (list,tuple)) else [strides[s]]*3
            self.up.append(nn.ConvTranspose3d(enc_ch, enc_ch, kernel_size=st, stride=st, bias=True))
            nc = dec_convs[i] if i < len(dec_convs) else 1
            dl = []
            for c in range(nc):
                dl.append(ConvBlock3D((enc_ch+skip_ch) if c==0 else out_ch, out_ch, 3))
            self.dec.append(nn.Sequential(*dl))
            self.seg.append(nn.Conv3d(out_ch, num_classes, 1))
        if aux_heads:
            self.sdf_head = nn.Conv3d(features[0], 1, 3, padding=1)
            self.topo_head = nn.Conv3d(features[0], 1, 3, padding=1)

    def forward(self, x):
        skips = []
        for i, enc in enumerate(self.enc):
            x = enc(x)
            skips.append(x)
        outputs = []
        x = skips[-1]
        for i, (u, d, s) in enumerate(zip(self.up, self.dec, self.seg)):
            skip = skips[-(i+2)]
            x = u(x)
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode="trilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = d(x)
            outputs.append(s(x))
        outputs = outputs[::-1]
        if self.deep_supervision and self.training:
            return outputs
        return outputs[0]


In [ ]:

# ============================================================
# Load Models (v3: handles aux head keys gracefully)
# ============================================================
models = {}
metas = {}
model_devices = {}  # mname → device; models distributed round-robin across all GPUs

_n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
_gpu_idx = 0
if _n_gpus > 1:
    print(f"[LOAD] {_n_gpus} GPUs detected — distributing models round-robin")

for mname, ckpt_path in MODEL_PATHS.items():
    print(f"\n[LOAD] {mname} from {ckpt_path}...")
    m = ResidualEncoderUNet(
        in_channels=IN_CHANNELS, num_classes=NUM_CLASSES,
        features=FEATURES, blocks=BLOCKS, strides=STRIDES,
        dec_convs=DEC_CONVS, deep_supervision=False, use_grad_ckpt=False,
        aux_heads=False  # No aux heads needed for inference
    )

    sd = torch.load(ckpt_path, map_location="cpu")
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]

    # Strip torch.compile prefix: checkpoints saved while compiled have "_orig_mod." on every key.
    # The inference model is not compiled at load time, so keys must be stripped first.
    if any(k.startswith("_orig_mod.") for k in sd):
        sd = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in sd.items()}
        print(f"[LOAD] Stripped _orig_mod. prefix (compiled checkpoint) for {mname}")

    # v3: Filter out aux head keys (sdf_head, topo_head) from checkpoint
    filtered_sd = {k: v for k, v in sd.items()
                   if not k.startswith("sdf_head.") and not k.startswith("topo_head.")}

    missing, unexpected = m.load_state_dict(filtered_sd, strict=False)
    if unexpected:
        # Filter out seg head mismatches from deep supervision
        critical_unexpected = [k for k in unexpected if ".seg." not in k]
        if critical_unexpected:
            raise RuntimeError(f"[FATAL] {mname}: {len(critical_unexpected)} UNEXPECTED keys!")
    if missing:
        non_seg_missing = [k for k in missing if ".seg." not in k]
        if non_seg_missing:
            raise RuntimeError(f"[FATAL] {mname}: {len(non_seg_missing)} critical missing keys!")

    # Round-robin GPU assignment: remembers which GPU each model will run on.
    # Models are kept in CPU RAM and moved to GPU one at a time during inference.
    # This keeps ~13.9 GB free per GPU → 128³ PRECOMP_GEMM workspace (10.1 GB) fits safely.
    # (All-GPU-resident approach left only 10.44 GB free → 34.17 GB workspace → OOM.)
    _dev = torch.device(f"cuda:{_gpu_idx % _n_gpus}") if torch.cuda.is_available() else torch.device("cpu")
    _gpu_idx += 1
    m = m.to('cpu').eval().half()   # store in CPU RAM; moved to GPU lazily per-inference
    models[mname] = m
    model_devices[mname] = _dev

    meta_path = META_PATHS.get(mname)
    if meta_path and os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        meta["ckpt_weight_mult"] = CKPT_MULTS.get(mname, 1.0)  # inject per-variant weight multiplier
        metas[mname] = meta
        print(f"  Meta: fold={meta.get('fold')}, score={meta.get('best_val_score', meta.get('combined_proxy', '?'))}, "
              f"tl={meta.get('tl', '?')}, th={meta.get('th', '?')}, temp={meta.get('temperature', '?')}, "
              f"ckpt_mult={meta['ckpt_weight_mult']:.2f}")
    else:
        metas[mname] = {"model": mname, "ckpt_weight_mult": CKPT_MULTS.get(mname, 1.0)}

    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"  {mname}: {n_params:.1f}M params")

# Auto-drop weak models
active_models = {}
for mname, m in models.items():
    meta = metas.get(mname, {})
    proxy = meta.get("best_val_score", meta.get("combined_proxy", 0.5))
    if proxy < AUTO_DROP_THRESH and len(models) > 1:
        print(f"  [DROP] {mname}: proxy={proxy:.4f} < {AUTO_DROP_THRESH}")
    else:
        active_models[mname] = m

if len(active_models) == 0:
    active_models = dict(models)

models = active_models
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

print(f"\n[LOAD] {len(models)} model(s) active for inference")


In [ ]:

# ============================================================
# Post-Processing v3: 26-CC everywhere, smart dust, multi-scale bridge killer
# ============================================================

def hysteresis_3d(prob, tl, th):
    """3D hysteresis thresholding with 26-connectivity."""
    struct26 = generate_binary_structure(3, 3)
    strong = prob >= th
    weak = prob >= tl
    cc, n = cc_label(weak, structure=struct26)
    if n == 0:
        return np.zeros_like(prob, dtype=np.uint8)
    strong_ids = np.unique(cc[strong])
    strong_ids = strong_ids[strong_ids != 0]
    return np.isin(cc, strong_ids).astype(np.uint8)


def smart_dust_removal(mask, probs=None, min_size=PP_DUST_MIN_3D, conf_threshold=0.7):
    """v3: Remove small components only if they don't contain high-confidence voxels."""
    if min_size <= 0:
        return mask
    struct26 = generate_binary_structure(3, 3)
    cc, n = cc_label(mask, structure=struct26)
    if n == 0:
        return mask
    sizes = np.bincount(cc.ravel())
    keep = np.zeros(n + 1, dtype=bool)
    for i in range(1, n + 1):
        if sizes[i] >= min_size:
            keep[i] = True
        elif probs is not None:
            # Keep small components with high-confidence voxels
            comp_mask = (cc == i)
            if probs[comp_mask].max() > conf_threshold:
                keep[i] = True
    return (keep[cc]).astype(np.uint8)


def fill_holes_2d(mask, max_area=PP_HOLE_MAX_2D):
    """v3: Cautious 2D hole filling — only fill if fully surrounded."""
    if max_area <= 0:
        return mask
    out = mask.copy()
    struct2d = generate_binary_structure(2, 1)
    for z in range(out.shape[0]):
        sl = out[z].astype(bool)
        bg = ~sl
        lbl, n = cc_label(bg, structure=struct2d)
        if n == 0:
            continue
        border = set()
        border.update(lbl[0, :].tolist())
        border.update(lbl[-1, :].tolist())
        border.update(lbl[:, 0].tolist())
        border.update(lbl[:, -1].tolist())
        for k in range(1, n + 1):
            if k in border:
                continue
            area = int((lbl == k).sum())
            if area <= max_area:
                sl[lbl == k] = True
        out[z] = sl.astype(np.uint8)
    return out


def xy_opening(mask, radius=PP_OPEN_R_XY):
    if radius <= 0:
        return mask
    struct2d = generate_binary_structure(2, 1)
    out = np.empty_like(mask)
    for z in range(mask.shape[0]):
        sl = mask[z].astype(bool)
        sl = binary_erosion(sl, structure=struct2d, iterations=radius)
        sl = binary_dilation(sl, structure=struct2d, iterations=radius)
        out[z] = sl.astype(np.uint8)
    return out


def opening_3d(mask, radius=PP_OPEN_R_3D):
    if radius <= 0:
        return mask
    struct26 = generate_binary_structure(3, 3)  # v3: 26-connectivity
    m = mask.astype(bool)
    m = binary_erosion(m, structure=struct26, iterations=radius)
    m = binary_dilation(m, structure=struct26, iterations=radius)
    return m.astype(np.uint8)


def thickness_bridge_killer(mask, probs=None, theta_neck=3.0, min_lobe_size=PP_BK_MIN_LOBE):
    """v3: Bridge cutting via local thickness + morphological skeleton.
    Brain doc algorithm L1:
    1. Compute local thickness (2 * distance_transform inside FG)
    2. Find thin neck voxels (thickness <= theta_neck)
    3. For each thin neck component, test if cutting improves topology
    4. Accept cut only if it splits into 2+ large components without explosion
    """
    struct26 = generate_binary_structure(3, 3)
    struct6 = generate_binary_structure(3, 1)
    mask_bool = mask.astype(bool)
    orig_sum = mask_bool.sum()
    if orig_sum == 0:
        return mask

    # Step 1: Local thickness map
    dt_fg = distance_transform_edt(mask_bool)
    thickness = 2.0 * dt_fg  # local thickness proxy

    # Step 2: Find thin neck candidates
    neck_mask = mask_bool & (thickness > 0) & (thickness <= theta_neck)
    if not neck_mask.any():
        return mask

    # Step 3: Evaluate each thin-neck connected component
    cc_neck, n_neck = cc_label(neck_mask, structure=struct26)
    if n_neck == 0:
        return mask

    # Pre-compute original component count
    _, n_orig = cc_label(mask_bool, structure=struct26)
    result = mask_bool.copy()
    cuts_made = 0

    for ni in range(1, min(n_neck + 1, 50)):  # cap iterations
        neck_comp = (cc_neck == ni)
        neck_size = neck_comp.sum()
        if neck_size < 3 or neck_size > 5000:  # too small or too big to be a neck
            continue

        # Temporarily remove this neck
        test_mask = result.copy()
        test_mask[neck_comp] = False

        # Check resulting topology
        cc_test, n_test = cc_label(test_mask, structure=struct26)
        sizes_test = np.bincount(cc_test.ravel())

        # Accept cut if:
        # - Creates exactly 2+ large components (separates real structures)
        # - Doesn't cause component explosion (n_test <= n_orig + 5)
        # - Doesn't remove too much volume
        n_large = sum(1 for s in sizes_test[1:] if s >= min_lobe_size)
        vol_kept = test_mask.sum() / max(orig_sum, 1)

        if n_large >= 2 and n_test <= n_orig + 5 and vol_kept >= 0.5:
            result = test_mask
            cuts_made += 1

    if cuts_made > 0:
        return result.astype(np.uint8)
    return mask


def multiscale_bridge_killer(mask, radii=(1, 2, 3), min_lobe_size=PP_BK_MIN_LOBE):
    """Fallback: Multi-scale erosion-based bridge killer."""
    struct6 = generate_binary_structure(3, 1)
    struct26 = generate_binary_structure(3, 3)

    best_result = mask.copy()
    orig_sum = mask.sum()
    if orig_sum == 0:
        return mask

    for radius in radii:
        eroded = binary_erosion(mask.astype(bool), structure=struct6, iterations=radius)
        if not eroded.any():
            continue

        cc_eroded, n_eroded = cc_label(eroded, structure=struct26)
        sizes = np.bincount(cc_eroded.ravel())

        result = np.zeros_like(mask, dtype=np.uint8)
        for i in range(1, n_eroded + 1):
            if sizes[i] < min_lobe_size:
                continue
            comp = (cc_eroded == i)
            grown = binary_dilation(comp, structure=struct6, iterations=radius)
            grown = grown & mask.astype(bool)
            result[grown] = 1

        if result.sum() >= orig_sum * 0.4:
            best_result = result
            break

    return best_result


def cavity_control_3d(mask, max_cavity_size=2000):  # was 500 — fill larger enclosed 3D cavities
    """Brain doc L3: Fill small enclosed cavities (k2 proxy).
    C = fill(P) - P. Fill only if |C_j| <= threshold."""
    try:
        struct26 = generate_binary_structure(3, 3)
        mask_bool = mask.astype(bool)
        # Fill ALL holes by flood-filling from borders
        from scipy.ndimage import binary_fill_holes
        filled = binary_fill_holes(mask_bool, structure=struct26)
        cavities = filled & ~mask_bool
        if not cavities.any():
            return mask
        # Only fill small cavities
        cc_cav, n_cav = cc_label(cavities, structure=struct26)
        result = mask.copy()
        filled_count = 0
        for i in range(1, n_cav + 1):
            comp = (cc_cav == i)
            if comp.sum() <= max_cavity_size:
                result[comp] = 1
                filled_count += 1
        return result
    except Exception:
        return mask


def _close_surface_gaps(mask, fused_prob, max_bridge_r=6, prob_thresh=0.25):
    """Aggressive surface gap-closure for 2-12 CC range.
    Progressively dilates the entire mask to find the minimum bridge radius that
    merges all CCs into one connected surface, then erodes back.
    Constrained: bridging voxels must have fused_prob > prob_thresh (surface-consistent).
    Called after Frangi+postprocess but before catastrophe_fallback."""
    struct26 = generate_binary_structure(3, 3)
    _, n_start = cc_label(mask.astype(bool), structure=struct26)
    if n_start <= 1:
        return mask   # already clean

    best_mask = mask.copy()
    best_n = n_start

    prob_ok = (fused_prob > prob_thresh)

    for r in range(1, max_bridge_r + 1):
        dilated = ndi.binary_dilation(mask.astype(bool), iterations=r)
        # Bridge only through high-probability voxels (surface-consistent)
        bridged = (dilated & prob_ok) | mask.astype(bool)
        eroded = ndi.binary_erosion(bridged, iterations=max(1, r - 1))
        # Keep original + bridged to avoid shrinking
        candidate = (eroded | mask.astype(bool)).astype(np.uint8)
        _, n_cand = cc_label(candidate.astype(bool), structure=struct26)
        if n_cand < best_n:
            best_n = n_cand
            best_mask = candidate
            if n_cand == 1:
                break  # fully connected, stop

    if best_n < n_start:
        print(f"    [CLOSE_GAPS] Surface gaps closed: {n_start} → {best_n} CCs")
    return best_mask


def anti_split_repair(mask, probs=None, max_gap=5, min_comp_size=500):
    """Brain doc L2: Careful fragment reconnection.
    Merge fragments only if: gap small AND thickness suggests same sheet.
    NEVER connect if it would create bridges across wraps."""
    struct26 = generate_binary_structure(3, 3)
    mask_bool = mask.astype(bool)
    cc, n = cc_label(mask_bool, structure=struct26)
    if n <= 1:
        return mask

    sizes = np.bincount(cc.ravel())
    # Find small fragments
    small_comps = [i for i in range(1, n + 1) if 10 < sizes[i] < min_comp_size]
    if not small_comps:
        return mask

    result = mask.copy()
    repaired = 0

    for ci in small_comps:
        comp = (cc == ci)
        # Find nearest large component via dilation
        dilated = ndi.binary_dilation(comp, iterations=max_gap)
        # Check which large components the dilation touches
        touched_labels = set(np.unique(cc[dilated & (cc != ci) & (cc > 0)])) - {0}
        large_touched = [l for l in touched_labels if sizes[l] >= min_comp_size]

        if len(large_touched) != 1:
            # Don't repair if touching 0 or 2+ large components (ambiguous/bridge risk)
            continue

        # Check gap region — only connect if probs are reasonably high in the gap
        gap_region = dilated & ~mask_bool
        if probs is not None and gap_region.any():
            gap_confidence = probs[gap_region].mean() if gap_region.any() else 0
            if gap_confidence < 0.3:
                continue  # Low confidence gap — don't connect

        # Connect via constrained dilation of the small component
        bridge = ndi.binary_dilation(comp, iterations=max_gap) & ~mask_bool
        # Only keep bridge voxels between the two components
        target_comp = (cc == large_touched[0])
        target_dilated = ndi.binary_dilation(target_comp, iterations=max_gap)
        bridge = bridge & target_dilated
        result[bridge] = 1
        repaired += 1

    return result


def postprocess(prob, tl, th, probs_for_dust=None, verbose=True, _idempotency_check=True,
                n_cc_pre=None):
    """v3 PODIUM: Full post-processing pipeline with brain doc algorithms.
    Upgrade 4: Pathology-gated bridge-kill and anti-split — only run aggressive steps
    when pathology indicators (CC count, fragmentation) actually require them."""
    tag = "    [PP]" if verbose else ""

    prob = gaussian_filter(prob, sigma=PP_SMOOTH_SIGMA).astype(np.float32)
    prob = np.clip(prob, 0, 1)

    mask = hysteresis_3d(prob, tl, th)
    if verbose:
        print(f"{tag} After hysteresis(tl={tl:.2f},th={th:.2f}): {mask.sum()} FG voxels")

    # Fallback for empty mask
    n_retries = 0
    retry_tl, retry_th = tl, th
    while mask.sum() == 0 and n_retries < 3:
        retry_tl = max(0.15, retry_tl - PP_EMPTY_TL_DROP)
        retry_th = max(0.30, retry_th - PP_EMPTY_TH_DROP)
        mask = hysteresis_3d(prob, retry_tl, retry_th)
        n_retries += 1

    # 3D opening
    if PP_OPEN_R_3D > 0:
        before = mask.sum()
        mask = opening_3d(mask, PP_OPEN_R_3D)
        if verbose and mask.sum() != before:
            print(f"{tag} 3D opening: {before} -> {mask.sum()}")

    # Upgrade 4: Pathology-gated bridge kill — only if CC count signals fragmentation
    struct26_pp = generate_binary_structure(3, 3)
    _, n_cc_now = cc_label(mask.astype(bool), structure=struct26_pp)
    # Use pre-computed CC count if caller provided it (from fused logit at th)
    _n_cc_check = n_cc_pre if n_cc_pre is not None else n_cc_now
    _run_bridge_kill = PP_BRIDGE_KILL and (_n_cc_check > PP_PATHOLOGY_CC_THRESH)
    if verbose:
        gate_tag = "ACTIVE" if _run_bridge_kill else f"SKIPPED (CC={_n_cc_check}<={PP_PATHOLOGY_CC_THRESH})"
        print(f"{tag} Bridge-kill gate: {gate_tag}")

    if _run_bridge_kill:
        before = mask.sum()
        try:
            mask = thickness_bridge_killer(mask, probs=probs_for_dust)
        except Exception:
            mask = multiscale_bridge_killer(mask)
        if verbose and mask.sum() != before:
            _, n_after = cc_label(mask.astype(bool), structure=struct26_pp)
            print(f"{tag} Bridge-kill: {before} -> {mask.sum()} ({n_after} components)")

    # v3: Smart dust removal (always run)
    before = mask.sum()
    mask = smart_dust_removal(mask, probs=probs_for_dust)
    if verbose and mask.sum() != before:
        print(f"{tag} Smart dust: {before} -> {mask.sum()}")

    # Brain doc L3: 3D cavity control
    before = mask.sum()
    mask = cavity_control_3d(mask)
    if verbose and mask.sum() != before:
        print(f"{tag} Cavity fill: {before} -> {mask.sum()}")

    mask = fill_holes_2d(mask)
    if PP_OPEN_R_XY > 0:
        mask = xy_opening(mask)

    # Upgrade 4: Pathology-gated anti-split — only if fragmented
    _, before_cc = cc_label(mask.astype(bool), structure=struct26_pp)
    _run_anti_split = (before_cc > 3) and (_n_cc_check >= 3)
    if _run_anti_split:
        before = mask.sum()
        mask = anti_split_repair(mask, probs=probs_for_dust)
        _, after_cc = cc_label(mask.astype(bool), structure=struct26_pp)
        if verbose and after_cc != before_cc:
            print(f"{tag} Anti-split: {before_cc} -> {after_cc} components")

    # Overfull guard
    fg_frac = mask.sum() / max(mask.size, 1)
    if fg_frac > PP_OVERFULL_FRAC:
        raised_th = th + PP_OVERFULL_TH_RAISE
        if verbose:
            print(f"{tag} Overfull ({fg_frac:.3f})! Re-thresholding with th={raised_th:.2f}")
        mask = hysteresis_3d(prob, tl, raised_th)
        mask = smart_dust_removal(mask)
        mask = fill_holes_2d(mask)

    # Playbook: Idempotency guard — postproc applied twice should give same result
    if _idempotency_check:
        mask2 = postprocess(prob, tl, th, probs_for_dust=probs_for_dust, verbose=False,
                            _idempotency_check=False, n_cc_pre=n_cc_pre)
        if not np.array_equal(mask, mask2):
            diff_vox = int(np.sum(mask != mask2))
            if verbose:
                print(f"{tag} WARNING: Postproc not idempotent! {diff_vox} voxels differ. Using stabilized result.")
            mask = mask2

    return mask


# Ensemble threshold computation
def _weighted_median(values, weights):
    if not values: return 0.5
    pairs = sorted(zip(values, weights))
    cumw = np.cumsum([w for _, w in pairs])
    total = cumw[-1]
    idx = np.searchsorted(cumw, total * 0.5)
    idx = min(idx, len(pairs) - 1)
    return pairs[idx][0]

_tls, _ths, _weights = [], [], []
for mn, meta in metas.items():
    if mn not in models: continue
    _tls.append(meta.get("tl", DEFAULT_TL))
    _ths.append(meta.get("th", DEFAULT_TH))
    proxy = meta.get("best_val_score", meta.get("combined_proxy", 0.5))
    _weights.append(max(float(proxy), 0.1))

if _weights:
    ENS_TL = _weighted_median(_tls, _weights)
    ENS_TH = _weighted_median(_ths, _weights)
else:
    ENS_TL, ENS_TH = DEFAULT_TL, DEFAULT_TH

PER_MODEL_TL = {mn: metas.get(mn, {}).get("tl", DEFAULT_TL) for mn in models}
PER_MODEL_TH = {mn: metas.get(mn, {}).get("th", DEFAULT_TH) for mn in models}

print(f"[PP] Ensemble thresholds: tl={ENS_TL:.3f}, th={ENS_TH:.3f}")


In [ ]:

# ============================================================
# Ensemble Sliding-Window Inference v3
# Upgrade 3: Logit-space Gaussian blending (seam-safe)
# Upgrade 2: Trimmed-mean logit fusion + per-model temperature
# Upgrade 5: Hard-volume adaptive TTA
# Upgrade 1: Learned per-volume threshold predictor
# ============================================================

def normalize_volume(vol, robust=True):
    """Brain doc: robust z-score with intensity clipping."""
    v = vol.astype(np.float32)
    lo, hi = np.percentile(v, [0.5, 99.5])
    v = np.clip(v, lo, hi)
    if robust:
        mu = np.median(v)
        sigma = np.median(np.abs(v - mu)) * 1.4826
    else:
        mu = v.mean()
        sigma = v.std()
    return (v - mu) / max(sigma, 1e-8)


def _gauss_1d(n):
    if n <= 1: return np.ones(n, dtype=np.float32)
    x = np.linspace(-1, 1, n, dtype=np.float32)
    return np.exp(-2.0 * x * x)


def _gauss_3d(shape):
    w = (_gauss_1d(shape[0])[:, None, None] *
         _gauss_1d(shape[1])[None, :, None] *
         _gauss_1d(shape[2])[None, None, :])
    return w / (w.max() + 1e-8)


def sharpen_probs(prob, temperature):
    if temperature == 1.0 or temperature <= 0:
        return prob
    p_clip = np.clip(prob, 1e-6, 1 - 1e-6)
    logit = np.log(p_clip / (1 - p_clip)) / temperature
    return 1.0 / (1.0 + np.exp(-logit))


@torch.no_grad()
def infer_single_model(vol_f32, model_m, roi, overlap, use_tta=False):
    """Sliding window inference. Returns FG logit map (D,H,W).
    Upgrade 3: accumulate logits (not probs) with Gaussian weights — seam-artifact free.
    Caller applies temperature + sigmoid once after this function."""
    D, H, W = vol_f32.shape
    rD, rH, rW = roi
    sD = max(1, int(rD * (1.0 - overlap)))
    sH = max(1, int(rH * (1.0 - overlap)))
    sW = max(1, int(rW * (1.0 - overlap)))
    w3d = _gauss_3d((rD, rH, rW))

    acc = np.zeros((D, H, W), dtype=np.float32)
    wacc = np.zeros((D, H, W), dtype=np.float32)

    z_starts = sorted(set(list(range(0, max(1, D-rD+1), sD)) + [max(0, D-rD)]))
    y_starts = sorted(set(list(range(0, max(1, H-rH+1), sH)) + [max(0, H-rH)]))
    x_starts = sorted(set(list(range(0, max(1, W-rW+1), sW)) + [max(0, W-rW)]))

    _model_device = next(model_m.parameters()).device  # use model's assigned GPU, not global DEVICE
    amp_on = (_model_device.type == "cuda")
    is_half = next(model_m.parameters()).dtype == torch.float16

    def _run_patch_logit(patch_np):
        """Run patch through model. Returns FG log-odds: logit[1] - logit[0]."""
        t = torch.from_numpy(patch_np[None, None].copy())
        if is_half: t = t.half()
        t = t.to(_model_device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=amp_on):
            logits = model_m(t)
            if isinstance(logits, (list, tuple)):
                logits = logits[0]
        lg = logits[0].float()   # shape (C, D, H, W)
        # log-odds FG vs BG — avoids softmax saturation
        logit_fg = (lg[1] - lg[0]).cpu().numpy()
        return logit_fg

    for z0 in z_starts:
        for y0 in y_starts:
            for x0 in x_starts:
                patch = vol_f32[z0:z0+rD, y0:y0+rH, x0:x0+rW]
                if patch.shape != (rD, rH, rW):
                    continue

                logit_fg = _run_patch_logit(patch)

                if use_tta:
                    n_tta = 1
                    # X, Y, Z axis flips — 3 axes × 2 orientations = 6 augmentations
                    # In-plane rotations removed (blueprint: avoid heavy TTA; 3 flips saves
                    # 3/7 ≈ 43% TTA compute vs original 7-way, preserving inference budget)
                    for ax in [2, 1, 0]:
                        flipped = np.flip(patch, axis=ax).copy()
                        lf = _run_patch_logit(flipped)
                        logit_fg = logit_fg + np.flip(lf, axis=ax)
                        n_tta += 1
                    logit_fg /= n_tta

                # Accumulate logits (Upgrade 3: logit-space blending, no seam artifacts)
                acc[z0:z0+rD, y0:y0+rH, x0:x0+rW] += logit_fg * w3d
                wacc[z0:z0+rD, y0:y0+rH, x0:x0+rW] += w3d

    wacc = np.maximum(wacc, 1e-8)
    return acc / wacc  # returns normalized logit map, NOT probability


def _difficulty_score(logit_map):
    """Upgrade 5: Volume difficulty for adaptive TTA.
    High score = complex topology, needs TTA. Low = easy, skip TTA."""
    prob = 1.0 / (1.0 + np.exp(-np.clip(logit_map, -50, 50)))
    fg_mask = (prob > 0.5)
    if not fg_mask.any():
        return 1.0  # empty = ambiguous = hard
    struct26 = generate_binary_structure(3, 3)
    _, n_cc = cc_label(fg_mask, structure=struct26)
    # Boundary voxels: proportion near decision boundary [0.3, 0.7]
    boundary_frac = float(np.mean((prob > 0.3) & (prob < 0.7)))
    # Fragmentation: CCs per unit FG volume
    fg_frac = float(fg_mask.mean())
    cc_norm = n_cc / max(fg_frac * 1000, 1)
    return float(np.clip(boundary_frac * 0.6 + min(cc_norm / 10.0, 0.4), 0.0, 1.0))


def _learned_threshold(prob, meta_coefs):
    """Upgrade 1: Use linear predictor coefficients from calibration meta.
    Features: [mean, std, p10, p25, p50, p75, p90, fg01, fg02, fg03, fg05].
    Returns (tl, th). Falls back to find_stable_threshold if no coefficients."""
    if not meta_coefs:
        return find_stable_threshold(prob)
    try:
        p_flat = prob.ravel()
        pcts = np.percentile(p_flat, [10, 25, 50, 75, 90])
        feats = np.array([
            float(p_flat.mean()),
            float(p_flat.std()),
            float(pcts[0]), float(pcts[1]), float(pcts[2]),
            float(pcts[3]), float(pcts[4]),
            float((p_flat > 0.1).mean()),
            float((p_flat > 0.2).mean()),
            float((p_flat > 0.3).mean()),
            float((p_flat > 0.5).mean()),
        ], dtype=np.float32)
        tl_coefs = np.array(meta_coefs.get("tl_coefs", []), dtype=np.float32)
        th_coefs = np.array(meta_coefs.get("th_coefs", []), dtype=np.float32)
        tl_intercept = float(meta_coefs.get("tl_intercept", 0.34))
        th_intercept = float(meta_coefs.get("th_intercept", 0.62))
        if len(tl_coefs) == len(feats) and len(th_coefs) == len(feats):
            tl = float(np.clip(np.dot(tl_coefs, feats) + tl_intercept, 0.18, 0.60))
            th = float(np.clip(np.dot(th_coefs, feats) + th_intercept, 0.38, 0.90))  # matches Nelder-Mead range
            if th > tl + 0.05:
                return tl, th
    except Exception:
        pass
    return find_stable_threshold(prob)


def risk_aware_fusion(per_model_probs, model_weights):
    """v3: Downweight models based on topology risk signals."""
    struct26 = generate_binary_structure(3, 3)
    names = list(per_model_probs.keys())
    adjusted = dict(model_weights)

    if len(names) < 2:
        return adjusted

    # Per-model topology risk assessment
    fg_fracs = {}
    comp_counts = {}
    for nm in names:
        binary = (per_model_probs[nm] > 0.5).astype(np.uint8)
        fg_fracs[nm] = float(binary.mean())
        if binary.sum() > 0:
            _, n_cc = cc_label(binary, structure=struct26)
            comp_counts[nm] = n_cc
        else:
            comp_counts[nm] = 0

    median_fg = float(np.median(list(fg_fracs.values())))
    median_cc = float(np.median(list(comp_counts.values()))) if comp_counts else 0

    for nm in names:
        # Near-empty mask — complete exclusion (0.15 weight still poisons trimmed mean)
        if fg_fracs[nm] < 1e-5 and median_fg > 1e-4:
            adjusted[nm] = 0.0
            print(f"    [RISK] {nm}: near-empty, EXCLUDED (weight -> 0)")
            continue

        # FG fraction outlier
        # Thresholds loosened (3.0→5.0, 0.33→0.20): the previous 3.0 threshold incorrectly
        # downweighted Model A (fg_ratio≈4.6, genuinely large surface) to 0.3× weight,
        # causing ensemble catastrophe. Plates can legitimately have 5–8× the median FG
        # when some models underpredict. Only penalise true outliers (>5× or <20% median).
        if median_fg > 1e-6:
            ratio = fg_fracs[nm] / max(median_fg, 1e-6)
            if ratio < 0.05 or ratio > 20.0:
                # Severely outlying (predicts essentially nothing or 20× too much) → exclude
                adjusted[nm] = 0.0
                print(f"    [RISK] {nm}: fg_ratio={ratio:.2f}, EXCLUDED (weight -> 0)")
                continue
            elif ratio > 5.0 or ratio < 0.20:
                adjusted[nm] = model_weights[nm] * 0.5
                print(f"    [RISK] {nm}: fg_ratio={ratio:.1f}, weight -> {adjusted[nm]:.3f}")

        # Fragmentation risk
        if median_cc > 0:
            cc_ratio = comp_counts[nm] / max(median_cc, 1)
            if cc_ratio > 5.0:
                adjusted[nm] = min(adjusted[nm], model_weights[nm] * 0.25)
                print(f"    [RISK] {nm}: {comp_counts[nm]} CCs (5x median), weight -> {adjusted[nm]:.3f}")
            elif cc_ratio > 3.0:
                adjusted[nm] = min(adjusted[nm], model_weights[nm] * 0.5)

    return adjusted


def trimmed_mean_logit_fusion(per_model_probs, trim_frac=0.15):
    """Upgrade 2: Trimmed-mean logit fusion. More robust than median for 3+ models.
    z_fuse(x) = trimmed_mean_m z_m(x) — drops top+bottom trim_frac of models per voxel.
    Falls back to median for n<=3 (trim would leave too few votes)."""
    names = list(per_model_probs.keys())
    logits_stack = []
    for nm in names:
        p = np.clip(per_model_probs[nm], 1e-6, 1 - 1e-6)
        logits_stack.append(np.log(p / (1 - p)))
    logits_arr = np.stack(logits_stack, axis=0)   # (N, D, H, W)
    n = len(names)
    if n <= 3:
        # Median for small ensembles (trimming would remove too many)
        return np.median(logits_arr, axis=0)
    # Trimmed mean: sort along model axis, drop top and bottom k models per voxel
    k = max(1, int(n * trim_frac))
    logits_sorted = np.sort(logits_arr, axis=0)
    trimmed = logits_sorted[k:n-k]   # shape (n-2k, D, H, W)
    return trimmed.mean(axis=0)


def find_stable_threshold(prob, tl_range=(0.25, 0.55), th_range=(0.40, 0.70), steps=6):
    """Brain doc: Threshold stability curve — find flat maxima region.
    Compute S(t) for grid of thresholds and pick the most stable."""
    best_score = -1.0
    best_tl, best_th = 0.40, 0.50
    best_stability = 0.0

    tl_grid = np.linspace(tl_range[0], tl_range[1], steps)
    th_grid = np.linspace(th_range[0], th_range[1], steps)
    scores = {}

    for tl in tl_grid:
        for th in th_grid:
            if th <= tl:
                continue
            mask = hysteresis_3d(prob, float(tl), float(th))
            fg_frac = mask.sum() / max(mask.size, 1)
            if fg_frac < 0.001 or fg_frac > 0.60:
                scores[(tl, th)] = 0.0
                continue
            struct26 = generate_binary_structure(3, 3)
            _, n_cc = cc_label(mask.astype(bool), structure=struct26)
            # Heuristic score: moderate FG fraction, few components
            s = max(0, 1.0 - abs(fg_frac - 0.15) * 3) * max(0, 1.0 - n_cc / 30.0)
            scores[(tl, th)] = s

    if not scores:
        return best_tl, best_th

    # Find flat maxima: best score with neighbors also high (stability)
    for (tl, th), s in scores.items():
        if s <= 0:
            continue
        # Check stability: how much does score change with small perturbation
        neighbors = []
        for dtl in [-0.05, 0, 0.05]:
            for dth in [-0.05, 0, 0.05]:
                key = (round(tl + dtl, 3), round(th + dth, 3))
                if key in scores:
                    neighbors.append(scores[key])
        if len(neighbors) >= 3:
            stability = min(neighbors) / max(s, 1e-8)
        else:
            stability = 0.5
        combined = s * 0.6 + stability * 0.4
        if combined > best_score:
            best_score = combined
            best_tl, best_th = float(tl), float(th)
            best_stability = stability

    return best_tl, best_th


def catastrophe_fallback(fused_mask, per_model_probs, model_roles):
    """Brain doc N: Catastrophe detection + fallback ladder.
    If fused result has explosion/bridge-risk, fall back to specialist output."""
    struct26 = generate_binary_structure(3, 3)
    _, n_cc_fused = cc_label(fused_mask.astype(bool), structure=struct26)

    # Surface detection: papyrus forms connected sheets. >25 CCs means the ensemble
    # fragmented the surface (gap artefacts from fusing misaligned model outputs).
    # Threshold raised from 20→25 because multi-scale Frangi reduces CCs; slight extra
    # tolerance avoids premature fallback to a single (potentially less accurate) model.
    fg_frac = fused_mask.sum() / max(fused_mask.size, 1)
    is_catastrophe = (n_cc_fused > 25) or (fg_frac < 0.005 and fg_frac > 0)

    if not is_catastrophe:
        return fused_mask

    print(f"    [CATASTROPHE] Detected: {n_cc_fused} components, fg_frac={fg_frac:.4f}")

    # Fallback ladder: "balanced" first so model_x variants are found before specialists
    fallback_order = ["balanced", "anti_merge", "generalist", "surface"]
    for role in fallback_order:
        for nm, r in model_roles.items():
            if r == role and nm in per_model_probs:
                prob = per_model_probs[nm]
                fb_mask = hysteresis_3d(prob, ENS_TL, ENS_TH)
                fb_mask = smart_dust_removal(fb_mask, probs=prob)
                _, n_cc_fb = cc_label(fb_mask.astype(bool), structure=struct26)
                fb_fg = fb_mask.sum() / max(fb_mask.size, 1)
                if n_cc_fb <= 15 and fb_fg > 0.01:
                    print(f"    [FALLBACK] Using {nm} ({role}): {n_cc_fb} components")
                    return fb_mask

    # Ultimate fallback: most conservative (highest threshold)
    print(f"    [FALLBACK] Using raised threshold as last resort")
    for nm in per_model_probs:
        prob = per_model_probs[nm]
        fb_mask = hysteresis_3d(prob, 0.45, 0.65)
        fb_mask = smart_dust_removal(fb_mask, probs=prob, min_size=512)
        _, n_cc = cc_label(fb_mask.astype(bool), structure=struct26)
        if n_cc <= 20:
            return fb_mask

    return fused_mask


def _frangi_surface_filter_single(prob_f64, sigma, alpha=0.5):
    """Single-scale 3D Frangi plate filter. Returns unnormalized surfaceness map.
    Uses analytical Cardano eigenvalue decomposition (fully vectorized)."""
    p = prob_f64
    Hzz = gaussian_filter(p, sigma, order=[2, 0, 0])
    Hyy = gaussian_filter(p, sigma, order=[0, 2, 0])
    Hxx = gaussian_filter(p, sigma, order=[0, 0, 2])
    Hzy = gaussian_filter(p, sigma, order=[1, 1, 0])
    Hzx = gaussian_filter(p, sigma, order=[1, 0, 1])
    Hyx = gaussian_filter(p, sigma, order=[0, 1, 1])
    q = (Hzz + Hyy + Hxx) / 3.0
    bz = Hzz - q;  by = Hyy - q;  bx = Hxx - q
    p2 = (bz**2 + by**2 + bx**2 + 2.0*(Hzy**2 + Hzx**2 + Hyx**2)) / 6.0
    p_sq = np.sqrt(np.maximum(p2, 0.0));  del p2
    inv_p = np.where(p_sq > 1e-12, 1.0 / p_sq, 0.0)
    cz = bz*inv_p;  cy = by*inv_p;  cx = bx*inv_p;  del bz, by, bx
    czy = Hzy*inv_p;  czx = Hzx*inv_p;  cyx = Hyx*inv_p
    del Hzz, Hyy, Hxx, Hzy, Hzx, Hyx, inv_p
    r = (cz*(cy*cx - cyx**2) - czy*(czy*cx - cyx*czx) + czx*(czy*cyx - cy*czx)) / 2.0
    del cz, cy, cx, czy, czx, cyx
    phi = np.arccos(np.clip(r, -1.0, 1.0)) / 3.0;  del r
    two_p = 2.0 * p_sq;  del p_sq
    # Unsorted eigenvalues from Cardano (ev_a=k0, ev_b=k1, ev_c=k2)
    ev_a = q + two_p * np.cos(phi)
    ev_b = q + two_p * np.cos(phi + 2.0*np.pi/3.0)
    ev_c = 3.0*q - ev_a - ev_b;  del q, two_p, phi
    # Sort by magnitude: |lam1| <= |lam2| <= |lam3|
    # For dark-on-bright sheet: lam3 is large-magnitude negative (normal to surface)
    # lam1, lam2 ~ 0  (tangent directions along the surface plane)
    abs_a, abs_b, abs_c = np.abs(ev_a), np.abs(ev_b), np.abs(ev_c)
    a_largest  = (abs_a >= abs_b) & (abs_a >= abs_c)
    b_largest  = (abs_b >  abs_a) & (abs_b >= abs_c)
    lam3 = np.where(a_largest, ev_a, np.where(b_largest, ev_b, ev_c))
    a_smallest = (abs_a <= abs_b) & (abs_a <= abs_c)
    b_smallest = (abs_b <  abs_a) & (abs_b <= abs_c)
    lam1 = np.where(a_smallest, ev_a, np.where(b_smallest, ev_b, ev_c))
    lam2 = ev_a + ev_b + ev_c - lam1 - lam3
    del ev_a, ev_b, ev_c, abs_a, abs_b, abs_c, a_largest, b_largest, a_smallest, b_smallest
    # Correct Frangi 1998 plate criterion: Ra = sqrt(lam1^2 + lam2^2) / |lam3|
    # Bug fix: original used sqrt(|lam1*lam2|)/|lam3| which fails to suppress tubes
    # (tube: lam1~0, lam2~lam3 → sqrt(|0*lam3|)/|lam3|=0, wrongly looks like plate)
    # Correct formula: sqrt(lam1^2+lam2^2)/|lam3| → tube gives |lam2|/|lam3|~1 (suppressed) ✓
    abs_l3 = np.abs(lam3)
    R_a = np.where(abs_l3 > 1e-12, np.sqrt(lam1**2 + lam2**2) / abs_l3, 1.0)
    S2 = lam1**2 + lam2**2 + lam3**2;  del lam1, lam2, abs_l3
    c2 = max(float(S2.max()) * 0.25, 1e-10)
    V = np.where(lam3 < 0,
        np.exp(-R_a**2 / (2.0*alpha**2)) * (1.0 - np.exp(-S2 / (2.0*c2))),
        0.0).astype(np.float32)
    del lam3, R_a, S2
    return V * float(sigma**2)   # Lindeberg scale-space normalization


def _frangi_surface_filter(prob, sigmas=(0.5, 1.0, 2.0), alpha=0.5):
    """Multi-scale 3D Frangi plate filter (host baseline technique, +0.019 LB).
    Identifies bright sheet-like structures via multi-scale Hessian eigenvalue analysis.
    Runs independently at each sigma and takes the maximum response (Lindeberg normalized).

    For papyrus surface (bright 2D sheet in 3D volume):
        lambda3 << 0  (large negative: strong curvature across the surface)
        lambda1, lambda2 ~ 0  (small: smooth variation along the surface plane)

    Returns surfaceness map normalized to [0, 1].
    Skips (returns zeros) if volume > 100M voxels to avoid CPU RAM OOM."""
    if prob.size > 100_000_000:
        print(f"    [FRANGI] Volume {prob.size/1e6:.0f}M voxels > 100M limit — skipping")
        return np.zeros_like(prob, dtype=np.float32)
    p_f64 = prob.astype(np.float64)
    best_V = np.zeros_like(prob, dtype=np.float32)
    for sigma in sigmas:
        try:
            V_s = _frangi_surface_filter_single(p_f64, sigma, alpha)
            best_V = np.maximum(best_V, V_s)
            del V_s
        except Exception as _se:
            print(f"    [FRANGI] sigma={sigma} failed: {_se}")
    del p_f64
    vmax = float(best_V.max())
    return (best_V / vmax) if vmax > 1e-10 else best_V


def logit_fuse_ensemble(per_model_probs, model_weights, model_roles=None, thresh_coefs=None):
    """v3 PODIUM: Trimmed-mean logit fusion + learned threshold + catastrophe fallback.
    Upgrade 2: trimmed-mean replaces median (more robust for large ensembles).
    Upgrade 1: learned threshold predictor replaces heuristic grid search."""
    names = list(per_model_probs.keys())
    n = len(names)

    if n == 1:
        fused_logit = np.log(np.clip(next(iter(per_model_probs.values())), 1e-6, 1-1e-6) /
                             np.clip(1 - next(iter(per_model_probs.values())), 1e-6, 1-1e-6))
        # Apply ensemble temperature to single model too
        if CONF_TEMPERATURE > 0 and CONF_TEMPERATURE != 1.0:
            fused_logit = fused_logit / CONF_TEMPERATURE
        prob = 1.0 / (1.0 + np.exp(-np.clip(fused_logit, -50, 50)))
        # Frangi surface filter (single-model path)
        # Blueprint: apply ONLY in uncertainty band [ENS_TL, ENS_TH) — not globally.
        try:
            _smap1 = _frangi_surface_filter(prob, sigmas=(0.5, 1.0, 2.0))
            if _smap1.max() > 1e-6:
                _prob1 = prob.copy()
                _s1_pos = _smap1 > 0.30;  _s1_neg = _smap1 < 0.05
                _bl1 = (_prob1 >= ENS_TL) & (_prob1 < ENS_TH)  # calibrated band, not hardcoded 0.44
                _prob1[_s1_pos & _bl1] = np.maximum(_prob1[_s1_pos & _bl1], ENS_TH + 0.01)
                _prob1[_s1_neg & (_prob1 >= ENS_TL) & (_prob1 < ENS_TH)] *= 0.6
                prob = np.clip(_prob1, 0.0, 1.0).astype(np.float32)
            del _smap1
        except Exception:
            pass
        return postprocess(prob, ENS_TL, ENS_TH, probs_for_dust=prob, verbose=True)

    # Step 1: Risk-aware weight adjustment
    vol_weights = risk_aware_fusion(per_model_probs, model_weights)

    # Step 2: Upgrade 2 — Trimmed-mean logit fusion on risk-filtered active models.
    # BUG FIX: vol_weights was computed above but trimmed_mean_logit_fusion previously
    # received ALL models (including weight=0 excluded ones), making risk_aware_fusion a
    # no-op for the multi-model case. Now filter to active models first, then fuse.
    active_probs = {nm: p for nm, p in per_model_probs.items()
                    if vol_weights.get(nm, 1.0) > 0.01}
    if len(active_probs) == 0:
        active_probs = per_model_probs  # safety: never exclude everything
        print("    [RISK] All models excluded — reverting to full ensemble")
    elif len(active_probs) < n:
        print(f"    [RISK] Active ensemble: {len(active_probs)}/{n} models (excluded {n - len(active_probs)})")

    n_active = len(active_probs)
    if n_active >= 2:
        result = trimmed_mean_logit_fusion(active_probs, trim_frac=0.15)
    else:
        # n_active == 1: weighted average (same as single-model but via vol_weights)
        w_total = sum(vol_weights[nm] for nm in active_probs)
        result = np.zeros_like(next(iter(active_probs.values())))
        for nm in active_probs:
            p = np.clip(active_probs[nm], 1e-6, 1 - 1e-6)
            result += np.log(p / (1 - p)) * (vol_weights[nm] / max(w_total, 1e-8))

    # Step 3: Ensemble temperature (applied after per-model temps already in probs)
    if CONF_TEMPERATURE != 1.0 and CONF_TEMPERATURE > 0:
        result = result / CONF_TEMPERATURE

    fused_prob = 1.0 / (1.0 + np.exp(-np.clip(result, -50, 50)))

    # Step 4: Upgrade 1 — Learned per-volume threshold predictor
    try:
        vol_tl, vol_th = _learned_threshold(fused_prob, thresh_coefs)
        print(f"    [THRESH] Predictor: tl={vol_tl:.3f}, th={vol_th:.3f}")
    except Exception:
        vol_tl, vol_th = ENS_TL, ENS_TH
        print(f"    [THRESH] Fallback: tl={vol_tl:.3f}, th={vol_th:.3f}")

    # Step 4.5: CC-minimizing threshold nudge.
    # If the initial threshold gives many fragments, try small nudges (±0.04, ±0.08)
    # to find a threshold that naturally produces fewer CCs without radically changing FG fraction.
    # Only activates when > 8 CCs detected — otherwise leave learned threshold untouched.
    try:
        _struct26_tn = generate_binary_structure(3, 3)
        _, _n_tn = cc_label((fused_prob > vol_th).astype(bool), structure=_struct26_tn)
        if _n_tn > 8:
            _fg_base_tn = float((fused_prob > vol_th).mean())
            _best_th_tn, _best_tl_tn, _best_n_tn = vol_th, vol_tl, _n_tn
            for _nudge in [-0.04, -0.08, 0.04, 0.08]:
                _th_c = float(np.clip(vol_th + _nudge, 0.28, 0.78))
                _tl_c = float(np.clip(vol_tl + _nudge, 0.14, _th_c - 0.06))
                _, _n_c = cc_label((fused_prob > _th_c).astype(bool), structure=_struct26_tn)
                _fg_c = float((fused_prob > _th_c).mean())
                # Accept only if: fewer CCs AND FG fraction doesn't change > 40%
                _fg_ok = (_fg_base_tn < 0.005) or (abs(_fg_c - _fg_base_tn) < 0.40 * _fg_base_tn)
                if _n_c < _best_n_tn and _fg_ok:
                    _best_n_tn, _best_th_tn, _best_tl_tn = _n_c, _th_c, _tl_c
            if _best_th_tn != vol_th:
                print(f"    [THRESH-NUDGE] CC-opt: {_n_tn}→{_best_n_tn} CCs, "
                      f"th={vol_th:.3f}→{_best_th_tn:.3f}")
                vol_th, vol_tl = _best_th_tn, _best_tl_tn
    except Exception:
        pass  # non-critical

    # Step 5: Uncertainty gating — suppress thin high-uncertainty connectors
    if n >= 2:
        prob_stack = np.stack([per_model_probs[nm] for nm in names], axis=0)
        uncertainty = np.var(prob_stack, axis=0)
        uncertain_mask = (uncertainty > 0.05) & (fused_prob > 0.3) & (fused_prob < 0.7)
        fused_prob[uncertain_mask] *= 0.7  # dampen uncertain connectors

    # Step 5.5: Frangi surface filter (host baseline technique, +~0.019 LB)
    # 3D Hessian eigenvalue analysis: bright plate-like (sheet) voxels get confirmed,
    # non-surface blobs get attenuated. Directly mirrors host postprocessing step.
    try:
        _t_fr = time.time()
        _smap = _frangi_surface_filter(fused_prob, sigmas=(0.5, 1.0, 2.0))
        if _smap.max() > 1e-6:
            _fp2 = fused_prob.copy()
            _surf_pos = _smap > 0.30   # strong plate-like surface response
            _surf_neg = _smap < 0.05   # definitely not surface
            # Blueprint: restrict Frangi boost/attenuation to uncertainty band [vol_tl, vol_th).
            # Outside this band the model is confident — Frangi would add noise, not signal.
            _borderline = (_fp2 >= vol_tl) & (_fp2 < vol_th)  # calibrated band, not hardcoded 0.44
            _fp2[_surf_pos & _borderline] = np.maximum(
                _fp2[_surf_pos & _borderline], vol_th + 0.01)
            # Attenuate: non-surface voxels inside uncertainty band only (conservative).
            _mid_blob = _surf_neg & (_fp2 >= vol_tl) & (_fp2 < vol_th)
            _fp2[_mid_blob] *= 0.6
            fused_prob = np.clip(_fp2, 0.0, 1.0).astype(np.float32)
            print(f"    [FRANGI] {time.time()-_t_fr:.1f}s | "
                  f"surface={float(_surf_pos.mean())*100:.1f}% | "
                  f"boost={int((_surf_pos & _borderline).sum())} | "
                  f"attenuate={int(_mid_blob.sum())}")
            del _fp2, _surf_pos, _surf_neg, _borderline, _mid_blob
        del _smap
    except Exception as _fe:
        print(f"    [FRANGI] Skipped ({type(_fe).__name__}: {_fe})")

    # Step 6: Postprocess with pathology gating
    struct26 = generate_binary_structure(3, 3)
    _, n_cc_pre = cc_label((fused_prob > vol_th).astype(bool), structure=struct26)
    mask = postprocess(fused_prob, vol_tl, vol_th, probs_for_dust=fused_prob,
                       verbose=True, n_cc_pre=n_cc_pre)

    # Step 6.5: Surface gap closure — connect nearby fragments before catastrophe check.
    # Runs only when 2-24 CCs exist (enough fragmentation to need help but not catastrophe).
    # Uses progressive morphological bridging constrained to high-probability regions.
    try:
        struct26_gc = generate_binary_structure(3, 3)
        _, n_before_gc = cc_label(mask.astype(bool), structure=struct26_gc)
        if 2 <= n_before_gc <= 24:
            mask = _close_surface_gaps(mask, fused_prob, max_bridge_r=5, prob_thresh=0.28)
    except Exception as _gce:
        pass  # non-critical, continue to catastrophe check

    # Step 7: Catastrophe fallback
    if model_roles is not None:
        mask = catastrophe_fallback(mask, per_model_probs, model_roles)

    return mask


def compute_auto_weights(model_names, metas_dict):
    """v3: Auto-weights from score-aligned proxy × role × ckpt_type multiplier."""
    raw = {}
    for mn in model_names:
        meta = metas_dict.get(mn, {})
        proxy = meta.get("best_val_score", meta.get("combined_proxy", 0.3))
        role = meta.get("model_role", "balanced")
        role_mult = ROLE_WEIGHTS.get(role, 1.0)
        ckpt_mult = float(meta.get("ckpt_weight_mult", 1.0))  # per-checkpoint type downweight
        raw[mn] = max(float(proxy) * role_mult * ckpt_mult, 0.05)

    vals = np.array([raw[mn] for mn in model_names])
    exp_vals = np.exp(vals - vals.max())
    weights = exp_vals / (exp_vals.sum() + 1e-8)
    weights = weights * len(model_names)
    weights = np.clip(weights, 0.1, None)
    return {mn: float(w) for mn, w in zip(model_names, weights)}


def ensemble_predict(vol_u8, use_tta):
    """Run all models with per-model ROI, then fuse.
    Upgrade 5: volume-difficulty-based adaptive TTA (not model-weakness-based).
    Upgrade 7: per-model temperature applied in logit space before sigmoid."""
    vol_f32 = normalize_volume(vol_u8)
    model_weights = compute_auto_weights(list(models.keys()), metas)

    per_model_logits = {}   # raw logit maps before temperature
    per_model_probs = {}    # temperature-scaled probs

    import threading, collections
    from concurrent.futures import ThreadPoolExecutor

    # GPU-grouped parallel inference: group models by their assigned device.
    # One thread per GPU runs its models serially → no concurrent allocation on the same device.
    # Both GPUs work simultaneously → ~2× wall-clock speedup vs pure serial.
    # Peak VRAM per GPU: model weights (955/764 MB) + one 128³ forward (~500 MB) << 14.56 GB.
    _gpu_groups = collections.defaultdict(list)
    for _mn in models:
        _gpu_groups[str(model_devices.get(_mn, DEVICE))].append(_mn)
    _n_gpu_threads = len(_gpu_groups)
    _logit_lock = threading.Lock()

    def _infer_gpu_group(mnames_for_gpu, use_tta_flag, tag):
        for _mn in mnames_for_gpu:
            _dev = model_devices.get(_mn, DEVICE)
            _m = models[_mn]
            # Move this model alone onto GPU; all others remain in CPU RAM.
            # With 1 model on GPU (191 MB), ~13.9 GB free → 128³ PRECOMP_GEMM workspace
            # (10.12 GB) fits with 3.8 GB to spare. 9-models-resident left only 10.44 GB → OOM.
            _m.to(_dev)
            _roi = tuple(metas.get(_mn, {}).get("patch_size", list(DEFAULT_ROI)))
            # Patch cap reads from _INF_PATCH_CAP_GLOBAL so OOM retry in main loop
            # can reduce it (128→96→64) and re-enter ensemble_predict at lower quality.
            _INF_PATCH_CAP = tuple(_INF_PATCH_CAP_GLOBAL)
            _roi = tuple(min(r, c) for r, c in zip(_roi, _INF_PATCH_CAP))
            _roi_c = tuple(min(r, s) for r, s in zip(_roi, vol_f32.shape))
            _ovlp = INF_OVERLAP  # always use config, not training meta (meta may store lower training overlap)
            _t0 = time.time()
            _lmap = infer_single_model(vol_f32, _m, _roi_c, _ovlp, use_tta_flag)
            _elapsed = time.time() - _t0
            _m.to('cpu')  # return to CPU RAM immediately
            # No synchronize: causes CUDA context poisoning via sticky VMM errors on T4
            # No empty_cache: causes allocator fragmentation; 10.12 GB workspace block stays
            # in PyTorch's caching allocator and is reused cleanly for the next model
            with _logit_lock:
                per_model_logits[_mn] = _lmap
                print(f"    {_mn} [{tag}]: {_elapsed:.1f}s roi={_roi_c} dev={_dev}")

    # Step 1: Scout pass — parallel across GPUs, serial within each GPU
    t0_all = time.time()
    _group_lists = list(_gpu_groups.values())
    with ThreadPoolExecutor(max_workers=_n_gpu_threads) as _pool:
        _futs = [_pool.submit(_infer_gpu_group, grp, False, "scout") for grp in _group_lists]
        for _f in _futs: _f.result()  # re-raise any exceptions

    # Step 2: TTA pass — always applied (TTA_DIFFICULTY_THRESH=0.0)
    did_tta = False
    if use_tta:
        w_total = sum(model_weights[mn] for mn in models)
        mean_logit = sum(per_model_logits[mn] * (model_weights[mn] / max(w_total, 1e-8))
                         for mn in models)
        difficulty = _difficulty_score(mean_logit)
        print(f"    [TTA] Volume difficulty: {difficulty:.3f} (thresh={TTA_DIFFICULTY_THRESH})")

        if difficulty > TTA_DIFFICULTY_THRESH and budget_ok(MAX_PRED_HOURS - 0.3):
            print(f"    [TTA] Applying full TTA — parallel across {_n_gpu_threads} GPU(s)")
            with ThreadPoolExecutor(max_workers=_n_gpu_threads) as _pool:
                _futs = [_pool.submit(_infer_gpu_group, grp, True, "+TTA") for grp in _group_lists]
                for _f in _futs: _f.result()
            did_tta = True
        else:
            print(f"    [TTA] Skipping (difficulty {difficulty:.3f} ≤ thresh {TTA_DIFFICULTY_THRESH})")

    # Step 3: Upgrade 7 — Apply per-model temperature in logit space, then sigmoid once
    for mname in models:
        logit_map = per_model_logits[mname]
        temp_cal = float(metas.get(mname, {}).get("temperature", 1.0))
        if temp_cal > 0 and temp_cal != 1.0:
            logit_map = logit_map / temp_cal   # scale logits (not probs)
        prob = 1.0 / (1.0 + np.exp(-np.clip(logit_map, -50, 50)))
        per_model_probs[mname] = prob
        w = model_weights[mname]
        tta_tag = " +TTA" if did_tta else ""
        print(f"    {mname}: T={temp_cal:.2f}, w={w:.3f}{tta_tag}")

    # Step 4: Collect ensemble threshold predictor coefficients from metas
    # Use the most recent (highest-proxy) model's coefficients if available
    thresh_coefs = None
    best_proxy = -1.0
    for mn in models:
        meta_mn = metas.get(mn, {})
        proxy = float(meta_mn.get("best_val_score", meta_mn.get("combined_proxy", 0.0)))
        if proxy > best_proxy and "thresh_coefs" in meta_mn:
            thresh_coefs = meta_mn["thresh_coefs"]
            best_proxy = proxy

    # Step 5: Fuse with trimmed-mean logit fusion + learned threshold
    roles = {mn: metas.get(mn, {}).get("model_role", "generalist") for mn in models}
    return logit_fuse_ensemble(per_model_probs, model_weights,
                               model_roles=roles, thresh_coefs=thresh_coefs)


In [ ]:

# ============================================================
# Main Inference Loop v3
# ============================================================
OUT_DIR = "/kaggle/working/preds"
os.makedirs(OUT_DIR, exist_ok=True)
ZIP_PATH = "/kaggle/working/submission.zip"

submission_meta = {
    "version": "v3.0",
    "models": list(models.keys()),
    "model_paths": {k: MODEL_PATHS[k] for k in models},
    "ens_mode": ENS_MODE,
    "ens_tl": ENS_TL, "ens_th": ENS_TH,
    "conf_temperature": CONF_TEMPERATURE,
    "default_roi": list(DEFAULT_ROI),
    "overlap": INF_OVERLAP,
    "pp_dust_min_3d": PP_DUST_MIN_3D,
    "pp_hole_max_2d": PP_HOLE_MAX_2D,
    "pp_bridge_kill": PP_BRIDGE_KILL,
    "auto_drop_thresh": AUTO_DROP_THRESH,
    "pytorch_version": torch.__version__,
    "metas": {k: {kk: vv for kk, vv in v.items()
                   if isinstance(vv, (int, float, str, bool))}
              for k, v in metas.items() if k in models},
}

n_test = len(test_ids)
time_per_vol_budget = (MAX_PRED_HOURS * 3600 - 300) / max(n_test, 1)
n_models = len(models)
print(f"[INF] {n_test} volumes, {time_per_vol_budget:.0f}s/vol budget, "
      f"{n_models} variants, overlap={INF_OVERLAP}, tta_thresh={TTA_DIFFICULTY_THRESH}")

# 6-level degradation ladder — applied proactively before each volume when
# rolling average × remaining_vols > remaining_budget × 0.80.
# Level 0 = best quality; each step sacrifices quality to preserve timing.
_DEGRADE_LADDER = [
    # (overlap, tta_eligible, patch_cap)
    (0.50,  True,  128),   # L0: full quality — TTA on difficult vols (gated by thresh)
    (0.50,  False, 128),   # L1: TTA completely off — 7x speedup vs L0
    (0.375, False, 128),   # L2: 40% fewer patches vs L1
    (0.25,  False, 128),   # L3: 64% fewer patches vs L0
    (0.25,  False,  96),   # L4: smaller patch window (lower VRAM, fewer patches)
    (0.25,  False,  64),   # L5: minimal patch window — last resort before empty
]
_degrade_level = [0]
_vol_rolling_times = []   # actual seconds per completed volume (last ≤6 kept for avg)

current_tta = USE_TTA
# Initial TTA check: if 5 variants × ~60s each already exceeds budget, start at L1.
_init_est_no_tta = n_models * 60
if _init_est_no_tta > time_per_vol_budget * 0.9:
    current_tta = False
    _degrade_level[0] = 1
    print(f"[INF] Starting at L1 (TTA off): est {_init_est_no_tta:.0f}s/vol "
          f"({n_models} variants×60s) vs budget {time_per_vol_budget:.0f}s")

# Pre-scan volume shapes so we can write correctly-shaped empty predictions
# for any volumes that can't be processed within the time budget.
# Uses tifffile metadata read (no decompression) — fast even for large volumes.
print("[INF] Pre-scanning volume shapes...")
vol_shapes = {}
for _vid, _vpath in zip(test_ids, test_paths):
    try:
        import tifffile as _tff
        with _tff.TiffFile(_vpath) as _tf:
            _pg = _tf.pages
            vol_shapes[_vid] = (len(_pg), int(_pg[0].shape[0]), int(_pg[0].shape[1]))
    except Exception:
        try:
            _v = read_tif(_vpath)
            vol_shapes[_vid] = _v.shape
            del _v
        except Exception:
            pass
print(f"[INF] Shape pre-scan: {len(vol_shapes)}/{n_test} volumes")

completed = 0
per_vol_stats = []

def _write_empty_vol(zf, vid, shape, reason="time_budget"):
    """Write a correctly-shaped all-zeros prediction into the zip."""
    try:
        empty = np.zeros(shape, dtype=np.uint8)
        _ep = os.path.join(OUT_DIR, f"{vid}.tif")
        write_tif(_ep, empty)
        zf.write(_ep, f"{vid}.tif")
        per_vol_stats.append({"id": vid, "time_s": 0, "fg_frac": 0.0, "error": reason})
        print(f"  [EMPTY] {vid}: wrote zero prediction ({reason})")
    except Exception as _ee:
        print(f"  [EMPTY-FAIL] {vid}: {_ee}")

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for vi, (vid, vpath) in enumerate(zip(test_ids, test_paths)):
        if not budget_ok(MAX_PRED_HOURS - 0.1):
            print(f"\n[TIME] Budget near limit. {completed}/{n_test} done.")
            print(f"[TIME] Writing empty predictions for {n_test - vi} remaining volumes.")
            for _vid, _vpath in zip(test_ids[vi:], test_paths[vi:]):
                if _vid in vol_shapes:
                    _write_empty_vol(zf, _vid, vol_shapes[_vid])
                else:
                    print(f"  [EMPTY-SKIP] {_vid}: no shape info, skipping")
            break

        t0 = time.time()
        print(f"\n[{vi+1}/{n_test}] {vid}...")
        orig_shape = None

        # Proactive degradation ladder: step down BEFORE attempting this volume if
        # rolling_avg_time × remaining_vols > remaining_budget × 0.80.
        # Uses rolling avg of last 6 volumes (more stable than single last-vol estimate).
        remaining = n_test - vi - 1
        if remaining > 0 and _vol_rolling_times:
            _rolling_avg = sum(_vol_rolling_times[-6:]) / len(_vol_rolling_times[-6:])
            _remain_budget = (MAX_PRED_HOURS - elapsed_h()) * 3600
            _est_remain = remaining * _rolling_avg
            if _est_remain > _remain_budget * 0.80 and _degrade_level[0] < len(_DEGRADE_LADDER) - 1:
                _next_level = _degrade_level[0] + 1
                _ovlp, _tta, _pcap = _DEGRADE_LADDER[_next_level]
                print(f"  [LADDER L{_degrade_level[0]}→{_next_level}] "
                      f"est {_est_remain:.0f}s > budget {_remain_budget*0.80:.0f}s: "
                      f"overlap={_ovlp}, tta={_tta}, pcap={_pcap}")
                INF_OVERLAP = _ovlp
                current_tta = _tta
                _INF_PATCH_CAP_GLOBAL[0] = _INF_PATCH_CAP_GLOBAL[1] = _INF_PATCH_CAP_GLOBAL[2] = _pcap
                _degrade_level[0] = _next_level

        # OOM-resilient inference: degrade patch cap and retry on CUDA OOM (up to 2 retries).
        # Ladder: 128³ (10.1 GB workspace) → 96³ (4.3 GB) → 64³ (1.3 GB).
        _oom_retries = 0
        _inference_done = False
        mask = None
        while not _inference_done:
            try:
                vol = read_tif(vpath)
                orig_shape = vol.shape
                print(f"  Shape: {orig_shape}, dtype: {vol.dtype}, cap={tuple(_INF_PATCH_CAP_GLOBAL)}")

                mask = ensemble_predict(vol, current_tta)

                assert mask.shape == orig_shape
                mask = mask.astype(np.uint8)
                mask = np.clip(mask, 0, 1)
                _inference_done = True

            except RuntimeError as _oom_e:
                _oom_msg = str(_oom_e).lower()
                if ("out of memory" in _oom_msg or "cuda" in _oom_msg) and _oom_retries < 2:
                    _oom_retries += 1
                    _new_cap = max(64, _INF_PATCH_CAP_GLOBAL[0] - 32)
                    print(f"  [OOM] Retry {_oom_retries}/2: patch cap {_INF_PATCH_CAP_GLOBAL[0]}→{_new_cap}, TTA off")
                    _INF_PATCH_CAP_GLOBAL[0] = _INF_PATCH_CAP_GLOBAL[1] = _INF_PATCH_CAP_GLOBAL[2] = _new_cap
                    current_tta = False
                    torch.cuda.empty_cache()
                    gc.collect()
                    orig_shape = None   # will be set on next read_tif
                else:
                    # Non-OOM error or retries exhausted → fall through to except
                    print(f"  [ERROR] {vid}: {_oom_e}")
                    traceback.print_exc()
                    _inference_done = True   # exit while loop, mask=None

            except Exception as e:
                print(f"  [ERROR] {vid}: {e}")
                traceback.print_exc()
                _inference_done = True   # exit while loop, mask=None

        # Write result or shape-correct empty fallback
        try:
            _shape_for_fallback = orig_shape if orig_shape is not None else vol_shapes.get(vid)
            if mask is not None:
                fg_frac = mask.sum() / mask.size
                if fg_frac == 0:
                    print(f"  [WARN] Empty prediction for {vid}!")
                out_path = os.path.join(OUT_DIR, f"{vid}.tif")
                write_tif(out_path, mask)
                zf.write(out_path, f"{vid}.tif")
                dt = time.time() - t0
                completed += 1
                _vol_rolling_times.append(dt)
                if len(_vol_rolling_times) > 12:   # keep last 12 for rolling avg
                    _vol_rolling_times.pop(0)
                per_vol_stats.append({"id": vid, "time_s": dt, "fg_frac": float(fg_frac),
                                      "shape": list(orig_shape), "degrade_level": _degrade_level[0]})
                print(f"  Done: {dt:.1f}s, FG={fg_frac:.4f}, L={_degrade_level[0]}")
            else:
                # Inference failed — write correctly-shaped empty prediction
                if _shape_for_fallback is not None:
                    _write_empty_vol(zf, vid, _shape_for_fallback, reason="inference_failed")
                    completed += 1
                else:
                    print(f"  [ERROR] {vid}: no shape info for fallback — volume skipped")
                    per_vol_stats.append({"id": vid, "time_s": 0, "fg_frac": 0.0,
                                          "error": "no_shape_for_fallback"})
        except Exception as _write_e:
            print(f"  [ERROR] Write failed for {vid}: {_write_e}")

        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

submission_meta["completed"] = completed
submission_meta["total_time_h"] = elapsed_h()
submission_meta["per_volume"] = per_vol_stats
meta_out = "/kaggle/working/submission_meta.json"
with open(meta_out, "w") as f:
    json.dump(submission_meta, f, indent=2)

print(f"\n[DONE] {completed}/{n_test} volumes in {elapsed_h():.2f}h")
print(f"  Submission: {ZIP_PATH}")


In [ ]:

# ============================================================
# Submission Validation
# ============================================================
assert os.path.exists(ZIP_PATH), f"submission.zip not found"

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()
    assert len(names) > 0, "submission.zip is empty"
    print(f"[ZIP] {len(names)} files in submission.zip")
    for n in names[:5]:
        info = zf.getinfo(n)
        print(f"  {n}: {info.file_size/1e6:.1f} MB")

    for n in names[:3]:
        with zf.open(n) as fp:
            header = fp.read(4)
            assert header[:2] in (b'II', b'MM'), f"{n}: Not a valid TIFF"

    zip_basenames = set(os.path.basename(n).replace(".tif", "") for n in names)
    missing = [tid for tid in test_ids if tid not in zip_basenames]
    if missing:
        print(f"  [WARN] Missing predictions for {len(missing)} volumes")
    else:
        print(f"  All {len(test_ids)} test volumes have predictions")

sz = os.path.getsize(ZIP_PATH) / 1e6
print(f"\n[OK] submission.zip: {sz:.1f} MB, {len(names)} volumes")
print(f"[OK] Total time: {elapsed_h():.2f}h")

if os.path.exists("/kaggle/working/submission_meta.json"):
    with open("/kaggle/working/submission_meta.json") as f:
        sm = json.load(f)
    print(f"\n[META] Version: {sm.get('version')}")
    print(f"[META] Models: {sm.get('models')}")
    stats = sm.get("per_volume", [])
    if stats:
        fg_fracs = [s["fg_frac"] for s in stats if "fg_frac" in s]
        if fg_fracs:
            print(f"[META] FG fraction: min={min(fg_fracs):.4f}, max={max(fg_fracs):.4f}, "
                  f"mean={np.mean(fg_fracs):.4f}")

gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
